In [2]:
import sys
sys.path.append("../")

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit.quantum_info import Statevector, SparsePauliOp
from qitp.engine.unitary_approximation import UnitaryApproximation
from qitp.engine.propagation import QITEPropagation
from qitp.engine.cqite_run import run_qite
from qitp.engine.prepare_state import constraint_rotation, prepare_state, density_from_rotations, support_only
from qitp.simulation.backend import SimulatorBackend

In [4]:
from qitp.ansatz.pauli_basis import (
    PauliBasis
)
from qitp.ansatz.commutator_basis import (
    CommutatorBasis
)

In [5]:
depth=2
dt=1e-2
nqubits = 6

In [6]:
nsteps=1000
target=0.7

energies=[]
fidelities=[]
errors=[]
magnetizations=[]

In [7]:
from qiskit.quantum_info import SparsePauliOp

Jzz = 0.3
Jxx = 0.3

n = nqubits

edges = [
    (1,2),
    (1,3),
    (2,3),
    (2,4),
    (3,4),
    (0,2),
    (0,4),
    (0,5),
    (4,5),
]

paulis = []
coeffs = []

for u, v in edges:
    s = ["I"] * n
    s[u] = "Z"
    s[v] = "Z"
    paulis.append("".join(s))
    coeffs.append(Jzz)

    s = ["I"] * n
    s[u] = "X"
    s[v] = "X"
    paulis.append("".join(s))
    coeffs.append(Jxx)

H = SparsePauliOp(paulis, coeffs=coeffs)

In [8]:
from qiskit.quantum_info import SparsePauliOp

n = nqubits

constraints = []

for i in range(n):
    s = ["I"] * n
    s[i] = "Y"
    constraints.append(
        SparsePauliOp(["".join(s)], coeffs=[1.0])
    )

Identity = SparsePauliOp(["I" * n], coeffs=[1.0])

In [9]:
Mz = constraints

In [10]:
H_eff = H + sum(Mz[1:], start=Mz[0])# + Identity

In [11]:
rotations = []

for constraint in Mz:

    rotations.append(
        constraint_rotation(
            constraint,
            target
        )
    )

In [12]:
rho = density_from_rotations(nqubits, rotations, degree=2)

seeds = support_only(rho)

In [13]:
basis = CommutatorBasis.from_seed(hamiltonian=H, seeds=seeds, depth=depth)

In [14]:
len(basis.paulis)

217

In [15]:
circuit = prepare_state(
    nqubits,
    rotations
)

In [16]:
circuit = circuit.decompose()

In [17]:
approx = UnitaryApproximation(basis, trot=2)

backend = SimulatorBackend()

prop = QITEPropagation(
    H,
    backend,
    approx
)

prop.constraints = [
    Identity]
prop.constraints.extend(Mz)

In [18]:
from qiskit.quantum_info import Statevector

psi = Statevector.from_instruction(circuit)

In [19]:
state = psi

In [ ]:
results = run_qite(
    prop,
    None,
    H,
    Mz,
    dt,
    nsteps,
    state #Remove for exponential advantage
)

QITE:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
energies = results["energies"]
magnetizations = results["magnetizations"] #[np.sum(m) for m in results["magnetizations"]]
final_circuit = results["circuit"]
final_state = results["state"]

In [ ]:
print(np.round(energies[-1],3))

In [ ]:
print(np.round(magnetizations[-1],3))

In [ ]:
plt.plot(energies); plt.title('Energy'); plt.grid(); plt.show()

In [ ]:
plt.plot(magnetizations); plt.title("Magnetization"); plt.grid(); plt.show()

In [ ]:
evals = np.linalg.eigvalsh(
    H.to_matrix()
)

print("Exact spectrum")

print(np.round(evals,2))

In [ ]:
Hm = H.to_matrix()

In [ ]:
C = []

for i in range(nqubits):

    p = ["I"] * nqubits
    p[i] = "Y"

    C.append(
        SparsePauliOp(
            ["".join(p)],
            coeffs=[1.0]
        ).to_matrix()
    )

I = np.eye(Hm.shape[0])

constraints_mat = [I]
constraints_mat.extend(C)

In [ ]:
def compute_betas_dense(
    psi,
    Hm,
    constraints
):

    m = len(constraints)

    gamma = np.empty(m)
    Gmat = np.empty((m, m))

    #
    # H|psi>
    #
    Hpsi = Hm @ psi

    #
    # B_i|psi> (calcolati una sola volta)
    #
    Bpsi = [
        B @ psi
        for B in constraints
    ]

    #
    # Gamma e Gram
    #
    for i in range(m):

        Bipsi = Bpsi[i]

        gamma[i] = -np.real(
            np.vdot(
                Hpsi,
                Bipsi
            )
        )

        #
        # Triangolo superiore
        #
        for j in range(i, m):

            value = np.real(
                np.vdot(
                    Bipsi,
                    Bpsi[j]
                )
            )

            Gmat[i, j] = value
            Gmat[j, i] = value

    return np.linalg.pinv(Gmat) @ gamma

In [ ]:
from scipy.sparse.linalg import expm_multiply

def cite_step(
    psi,
    Hm,
    constraints_mat,
    dt,
    nsub=100
):

    h = dt / nsub

    for _ in range(nsub):

        beta = compute_betas_dense(
            psi,
            Hm,
            constraints_mat
        )

        G = Hm.copy()

        for b, B in zip(beta, constraints_mat):
            G += b * B

        psi = expm_multiply(-h * G, psi)
        psi /= np.linalg.norm(psi)

    return psi

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

energies_exact = []
constraints_history = [[] for _ in C]

append_energy = energies_exact.append
append_constraints = [hist.append for hist in constraints_history]

psi_exact = psi.data

for step in tqdm(range(nsteps), desc="Exact CITE"):

    psi_exact = cite_step(
        psi_exact,
        Hm,
        constraints_mat,
        dt=dt
    )

    append_energy(
        np.real(
            np.vdot(
                psi_exact,
                Hm @ psi_exact
            )
        )
    )

    for append, B in zip(append_constraints, constraints_mat):

        append(
            np.real(
                np.vdot(
                    psi_exact,
                    B @ psi_exact
                )
            )
        )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

energies = np.asarray(energies)
energies_exact = np.asarray(energies_exact)

iters = np.arange(len(energies))

fig, ax = plt.subplots(
    figsize=(8, 3.8),
    dpi=100
)

ax.plot(
    iters,
    energies_exact,
    lw=2.8,
    label="Exact CITE"
)

ax.plot(
    iters,
    energies,
    lw=2.5,
    ls="--",
    label="CQITE"
)

ax.set_xlabel("Iteration", fontsize=12)
ax.set_ylabel(r"$\langle H\rangle$", fontsize=12)

ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.tick_params(direction="in")

plt.tight_layout()
fig.savefig("zzxxy_exp.pdf", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
mag.shape

In [ ]:
mag = np.asarray(magnetizations)

fig, ax = plt.subplots(
    figsize=(8, 3.8)
)

eps = 5e-2

for k in range(mag.shape[1]):

    rel_error = np.abs((mag[:, k] - target) / target)

    ax.plot(
        rel_error * (1 + k * eps),
        lw=2,
        label=rf"$C_{k+1}$"
    )

ax.set_yscale("log")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Relative error")
ax.grid(alpha=.3)

ax.legend()

plt.tight_layout()

fig.savefig(
    "zzxxy_con.pdf",
    bbox_inches="tight",
    transparent=True
)

In [ ]:
mag = np.asarray(magnetizations)

fig, ax = plt.subplots(
    figsize=(8,3.8)
)

for k in range(mag.shape[1]):
    ax.plot(
        np.abs(mag[:,k]),
        lw=2,
        label=rf"$C_{k+1}$"
    )

#ax.set_yscale("log")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"$\langle C\rangle$")
ax.grid(alpha=.3)

ax.legend()

plt.tight_layout()

fig.savefig("zzxxy_mag.pdf", bbox_inches="tight", transparent=True)